# NB03: Data Analysis

**Author:** martinezmerino

This notebook explores the tidy team-match table from NB02 to answer: **is home advantage equally strong across Europe's big five leagues, or does it vary?** Analysis is exploratory - I'm comparing groups and checking consistency, not fitting a predictive or inferential model.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

df = pd.read_csv("data/processed/matches_team_level.csv")
df.head()

,league,league_code,season,match_id,date,matchday,team,opponent,venue,goals_for,goals_against,goal_diff,result,points
0,Premier League,PL,2025,537785,2025-08-15,1,Liverpool FC,AFC Bournemouth,home,4,2,2,win,3
1,Premier League,PL,2025,537785,2025-08-15,1,AFC Bournemouth,Liverpool FC,away,2,4,-2,loss,0
2,Premier League,PL,2025,537786,2025-08-16,1,Aston Villa FC,Newcastle United FC,home,0,0,0,draw,1
3,Premier League,PL,2025,537786,2025-08-16,1,Newcastle United FC,Aston Villa FC,away,0,0,0,draw,1
4,Premier League,PL,2025,537787,2025-08-16,1,Brighton & Hove Albion FC,Fulham FC,home,1,1,0,draw,1


## Checking the comparison is fair before I start

Before comparing home and away performance, I check that every team played the same number of home and away matches. If a team had more home fixtures than away ones, it could look like it "benefits" from home advantage just because of the schedule, not because it actually performs differently at home.

In [2]:
venue_counts = df.groupby(["league", "team", "venue"]).size().unstack("venue")
venue_counts["imbalance"] = (venue_counts["home"] - venue_counts["away"]).abs()

print("Teams with unequal home/away match counts:")
print(venue_counts[venue_counts["imbalance"] > 0])
print(f"\nMax home/away imbalance across all {len(venue_counts)} teams: {venue_counts['imbalance'].max()}")

Teams with unequal home/away match counts:
venue                away  home  imbalance
league  team                              
Ligue 1 FC Nantes      17    16          1
        Toulouse FC    16    17          1

Max home/away imbalance across all 96 teams: 1


Two teams, FC Nantes and Toulouse FC (Ligue 1), are off by one match. Checking the full match list for that fixture (not just the `FINISHED` ones I originally pulled) shows why: their matchday-34 meeting was decided administratively (`status: AWARDED`, not an on-pitch result), while the first leg was played normally and finished 2-2. Since an awarded match doesn't reflect what happened on the pitch, excluding it from a home-advantage analysis is the right call - it just means these two teams have 33 matches instead of 34, a negligible imbalance out of 96 teams.

## Defining "home advantage"

I measure home advantage per team as the gap in points per game (PPG) between its home fixtures and its away fixtures: `home_ppg - away_ppg`. A team with a gap of zero gets no measurable benefit from playing at home; a large positive gap means home matches are systematically more rewarding than away ones.

I compute this per team first, and only then summarise by league, so I can check whether a league's average gap is broad-based across most of its teams or driven by a handful of outliers - a check I skipped on my midterm and had to add after the fact.

In [3]:
team_venue = (
    df.groupby(["league", "team", "venue"])
    .agg(
        matches=("points", "size"),
        points=("points", "sum"),
        goal_diff=("goal_diff", "sum"),
        wins=("result", lambda s: (s == "win").sum()),
    )
    .reset_index()
)
team_venue["ppg"] = team_venue["points"] / team_venue["matches"]
team_venue["win_rate"] = team_venue["wins"] / team_venue["matches"]
team_venue["goal_diff_per_game"] = team_venue["goal_diff"] / team_venue["matches"]

team_wide = team_venue.pivot(index=["league", "team"], columns="venue", values=["ppg", "win_rate", "goal_diff_per_game"])
team_wide.columns = [f"{stat}_{venue}" for stat, venue in team_wide.columns]
team_wide = team_wide.reset_index()

team_wide["ppg_gap"] = team_wide["ppg_home"] - team_wide["ppg_away"]
team_wide["win_rate_gap"] = team_wide["win_rate_home"] - team_wide["win_rate_away"]
team_wide["goal_diff_gap"] = team_wide["goal_diff_per_game_home"] - team_wide["goal_diff_per_game_away"]

team_wide.sort_values("ppg_gap", ascending=False).head(10)

,league,team,ppg_away,ppg_home,win_rate_away,win_rate_home,goal_diff_per_game_away,goal_diff_per_game_home,ppg_gap,win_rate_gap,goal_diff_gap
22,La Liga,Elche CF,0.421053,1.842105,0.052632,0.473684,-1.000000,0.578947,1.421053,0.421053,1.578947
29,La Liga,RCD Mallorca,0.473684,1.736842,0.105263,0.473684,-1.052632,0.526316,1.263158,0.368421,1.578947
20,La Liga,Club Atlético de Madrid,1.210526,2.421053,0.315789,0.789474,-0.210526,1.157895,1.210526,0.473684,1.368421
19,La Liga,CA Osasuna,0.526316,1.684211,0.105263,0.473684,-0.684211,0.368421,1.157895,0.368421,1.052632
23,La Liga,FC Barcelona,1.947368,3.000000,0.631579,1.000000,0.631579,2.473684,1.052632,0.368421,1.842105
37,La Liga,Villarreal CF,1.368421,2.421053,0.368421,0.789474,-0.157895,1.526316,1.052632,0.421053,1.684211
13,Bundesliga,SC Freiburg,0.882353,1.882353,0.235294,0.529412,-1.000000,0.647059,1.000000,0.294118,1.647059
65,Premier League,Fulham FC,0.894737,1.842105,0.210526,0.578947,-0.736842,0.526316,0.947368,0.368421,1.263158
16,Bundesliga,VfB Stuttgart,1.352941,2.294118,0.352941,0.705882,0.470588,0.823529,0.941176,0.352941,0.352941
66,Premier League,Leeds United FC,0.789474,1.684211,0.105263,0.473684,-0.789474,0.421053,0.894737,0.368421,1.210526


## Does the gap hold across most teams in each league, or just a few?

In [4]:
league_summary = (
    team_wide.groupby("league")
    .agg(
        teams=("team", "size"),
        mean_ppg_gap=("ppg_gap", "mean"),
        median_ppg_gap=("ppg_gap", "median"),
        teams_favoured_at_home=("ppg_gap", lambda s: (s > 0).sum()),
        teams_favoured_away=("ppg_gap", lambda s: (s < 0).sum()),
    )
    .reset_index()
)
league_summary["pct_teams_favoured_at_home"] = 100 * league_summary["teams_favoured_at_home"] / league_summary["teams"]
league_summary.sort_values("mean_ppg_gap", ascending=False)

,league,teams,mean_ppg_gap,median_ppg_gap,teams_favoured_at_home,teams_favoured_away,pct_teams_favoured_at_home
1,La Liga,20,0.671053,0.631579,19,1,95.000000
2,Ligue 1,18,0.498366,0.558824,17,1,94.444444
3,Premier League,20,0.378947,0.526316,15,4,75.000000
0,Bundesliga,18,0.362745,0.411765,15,3,83.333333
4,Serie A,20,0.118421,0.131579,13,6,65.000000


## Chart 1: the home-advantage gap, team by team

A bar chart of the league averages would hide how spread out teams are within each league, so I plot every team's individual gap as a point (a strip plot), with a diamond marking each league's mean. This shows both the central tendency and how consistent - or not - the effect is.

In [5]:
fig1 = px.strip(
    team_wide,
    x="league",
    y="ppg_gap",
    color="league",
    hover_data=["team"],
    title="Home advantage (home PPG minus away PPG) by team, big five leagues 2025-26",
)
fig1.add_hline(y=0, line_dash="dash", line_color="gray")

for league, row in league_summary.set_index("league").iterrows():
    fig1.add_scatter(
        x=[league],
        y=[row["mean_ppg_gap"]],
        mode="markers",
        marker=dict(symbol="diamond", size=14, color="black"),
        name=f"{league} mean",
        showlegend=False,
    )

fig1.update_layout(yaxis_title="Home PPG - Away PPG", xaxis_title=None, showlegend=False)
fig1.write_image(str(FIGURES_DIR / "ppg_gap_by_league.png"), width=900, height=550, scale=2)
fig1.show()

## Chart 2: is the points gap backed up by goals, or just lucky results?

If home advantage is real rather than a quirk of close results, a team's gap in points per game should line up with its gap in goal difference per game. I check this per team, across all five leagues at once.

In [6]:
correlation = team_wide[["ppg_gap", "goal_diff_gap"]].corr().iloc[0, 1]
print(f"Correlation between PPG gap and goal-difference-per-game gap: {correlation:.2f}")

fig2 = px.scatter(
    team_wide,
    x="goal_diff_gap",
    y="ppg_gap",
    color="league",
    hover_data=["team"],
    title="Points gap vs goal-difference gap (home minus away), per team",
)
fig2.add_hline(y=0, line_dash="dash", line_color="gray")
fig2.add_vline(x=0, line_dash="dash", line_color="gray")
fig2.update_layout(xaxis_title="Goal-difference-per-game gap (home - away)", yaxis_title="PPG gap (home - away)")
fig2.write_image(str(FIGURES_DIR / "ppg_gap_vs_goal_diff_gap.png"), width=900, height=550, scale=2)
fig2.show()

Correlation between PPG gap and goal-difference-per-game gap: 0.82


## Summary numbers for the write-up

In [7]:
summary = league_summary.sort_values("mean_ppg_gap", ascending=False).reset_index(drop=True)
for _, row in summary.iterrows():
    print(
        f"{row['league']}: mean home-away PPG gap = {row['mean_ppg_gap']:.2f}, "
        f"{row['teams_favoured_at_home']:.0f}/{row['teams']:.0f} teams favoured at home "
        f"({row['pct_teams_favoured_at_home']:.0f}%)"
    )

print(f"\nOverall correlation between PPG gap and goal-diff gap: {correlation:.2f}")
print(f"League with strongest home advantage: {summary.iloc[0]['league']}")
print(f"League with weakest home advantage: {summary.iloc[-1]['league']}")

La Liga: mean home-away PPG gap = 0.67, 19/20 teams favoured at home (95%)
Ligue 1: mean home-away PPG gap = 0.50, 17/18 teams favoured at home (94%)
Premier League: mean home-away PPG gap = 0.38, 15/20 teams favoured at home (75%)
Bundesliga: mean home-away PPG gap = 0.36, 15/18 teams favoured at home (83%)
Serie A: mean home-away PPG gap = 0.12, 13/20 teams favoured at home (65%)

Overall correlation between PPG gap and goal-diff gap: 0.82
League with strongest home advantage: La Liga
League with weakest home advantage: Serie A
